In [4]:
import requests
import zipfile
import os

url =  'http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip'
filename = 'spa-eng.zip'
os.makedirs('../data', exist_ok=True)
zip_path = os.path.join('../data', filename)

try:
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f'File {filename} succesfullly donwloaded')
    else:
        print(f'Download failed. status: {response.status_code}')

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('../data')
        print('unzipped file')

except requests.exceptions.RequestException as e:
    print(f'HTTP request error: {e}')
except Exception as e:
    print(f'error: {e}')

File spa-eng.zip succesfullly donwloaded
unzipped file


In [5]:
text_file = '../data/spa-eng/spa.txt'
with open(text_file, encoding='utf-8') as f:
    lines = f.read().split('\n')[:-1]
text_pairs = []

for line in lines:
    english, spanish = line.split('\t')
    spanish = '[start]'+spanish+'[end]'
    text_pairs.append((english, spanish))
    
print(text_pairs[:5])

[('Go.', '[start]Ve.[end]'), ('Go.', '[start]Vete.[end]'), ('Go.', '[start]Vaya.[end]'), ('Go.', '[start]Váyase.[end]'), ('Hi.', '[start]Hola.[end]')]


In [7]:
import random
print(random.choice(text_pairs))

('Tom had no choice but to quit his job.', '[start]Tom no tuvo más remedio que dejar su trabajo.[end]')


In [8]:
random.shuffle(text_pairs)
num_val_samples = int(0.15*len(text_pairs))
num_train_samples = len(text_pairs) - 2*num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples+num_val_samples]
test_pairs = text_pairs[num_train_samples+num_val_samples:]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import string
import re

strip_chars = string.punctuation+'¿'
strip_chars = strip_chars.replace('[','')
strip_chars = strip_chars.replace(']','')

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size = 15000
sequence_length = 20
source_vectorization = layers.TextVectorization(
 max_tokens=vocab_size,
 output_mode="int",
 output_sequence_length=sequence_length,
)
target_vectorization = layers.TextVectorization(
 max_tokens=vocab_size,
 output_mode="int",
 output_sequence_length=sequence_length + 1,
 standardize=custom_standardization,
)
train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts) 

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~¿
